# Week 5: In-class warm-up problems

**Course:** AI for Aerospace & Mechanical Engineering (PhD), Block 1
**Format:** 4 problems, 5 to 10 minutes each, pen and paper first, then verify in Python.

> **Provenance:** new content. Missing-data mechanisms after Little and Rubin, *Statistical
> Analysis with Missing Data* (2019), Ch. 1; the z-score bound from Samuelson, *JASA* 63 (1968);
> Tukey's fences and the Hampel filter as in the Week 5 lab; leakage causes rewritten from
> GeeksforGeeks ("Data Leakage"); scaling formulas from W3Schools ("Python Machine Learning
> Scaling").

## Problem 1 (10 min): a dust echo in seven LiDAR readings

<div dir="rtl">مسئله ۱: در هفت خوانش LiDAR یک پژواک گردوغبار هست. با امتیاز z، حصارهای توکی و امتیاز مقاوم (میانه و MAD) آن را بررسی کنید و بگویید کدام روش روی ربات قابل اجراست.</div>

Seven consecutive readings while the wall is about 8 m away (metres):

`8.02, 7.98, 8.01, 1.20, 7.97, 8.03, 7.99`

a) Compute the mean, the population standard deviation and the $z$-score of 1.20. Flagged at $|z| > 3$?
b) Show that no $|z|$ can exceed $\sqrt{n-1}$ here. What does that say about rule a)?
c) Compute the median, the MAD and the robust score $|x - \tilde x|/(1.4826\,\mathrm{MAD})$ of 1.20.
d) Compute Tukey's fences with $k = 1.5$ (quartiles 7.975 and 8.015, as pandas computes them).
e) Which rule would you run on the rover, sample by sample, and over which window?

## Problem 2 (10 min): why is it missing?

<div dir="rtl">مسئله ۲: سازوکار گم‌شدن داده (MCAR، MAR یا MNAR) را در هر سناریو تعیین کنید و سپس سوگیری جایگذاری با میانگین را برای یک حسگر دمای اشباع‌شونده با دست حساب کنید.</div>

a) Classify each case as MCAR, MAR or MNAR, with one sentence of reasoning:
   1. The forward LiDAR returns nothing when the wall is beyond 12 m.
   2. 0.7 % of CAN-bus packets are lost at random; wheel speed is NaN for those samples.
   3. GPS fixes drop out inside the hangar, and the log has a column "in_hangar" (yes/no).
   4. A battery-temperature sensor writes an empty cell whenever the true temperature exceeds 60 °C.
b) Case 4. True temperatures over six minutes: 52, 55, 58, 61, 63, 57 °C. What does the log
   show? Compute the true mean, the mean of the logged values, and the mean after filling the
   blanks with the logged mean.
c) The thermal-protection rule is "derate the motor above 60 °C". After mean imputation, how
   many derating events does the cleaned log contain? What would you do instead?

## Problem 3 (10 min): leakage hunt

<div dir="rtl">مسئله ۳: در قطعه‌کد زیر پنج نشت داده پنهان است. هر کدام را پیدا کنید و اصلاحش را با کلاس‌های کتابخانه‌ی درس بنویسید. سپس احتمال نشت در تقسیم تصادفی را با دست حساب کنید.</div>

```python
log = pd.read_csv("rover_log.csv")
log["range_smooth"] = log["lidar_range_m"].rolling(21, center=True).mean()      # (A)
log["range_ahead_m"] = log["lidar_range_m"].shift(-10)                           # (B)
log = log.fillna(log.median())                                                   # (C)
log[FEATURES] = (log[FEATURES] - log[FEATURES].mean()) / log[FEATURES].std()      # (D)
train, test = train_test_split(log, test_size=0.25, shuffle=True)                # (E)
```

a) Name the leak in each of lines A to E and the fix.
b) With a random 25 % test split of a 10 Hz log, what is the probability that a test sample has
   **at least one** of its two immediate neighbours (±0.1 s) in the training set?

## Problem 4 (5 to 10 min): scale the test set with the training numbers

<div dir="rtl">مسئله ۴: سه مقیاس‌گذار را فقط با داده‌ی آموزش برازش دهید و دو نمونه‌ی آزمون را با دست مقیاس کنید؛ سپس اثر برازش روی همه‌ی داده‌ها را ببینید.</div>

Training ranges (m): 2, 4, 6, 8, 10. Test ranges: 12, 3.

a) z-score: compute $\mu$ and the population $\sigma$ of the training values, then $z$ of both test values.
b) min-max: the scaled test values. Is a value above 1 an error?
c) robust: median and IQR (pandas quartiles 4 and 8), then the scaled test values.
d) Repeat a) with $\mu, \sigma$ computed from all seven values. Why does the test value 12 now
   look less unusual, and why is that a problem for a monitoring system?

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd


def find_repo_root(start: Path) -> Path:
    """Walk up from ``start`` until the folder that contains ``src/aerospace_ai``."""
    for folder in (start, *start.parents):
        if (folder / "src" / "aerospace_ai").is_dir():
            return folder
    raise FileNotFoundError("Open this notebook from inside the course repository.")


REPO_ROOT = find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT / "src"))

from aerospace_ai.pipeline import HampelFilter, Standardize  # noqa: E402
from aerospace_ai.preprocessing import ImputeMedian, MinMaxScale, RobustScale  # noqa: E402

# Scratch space for verifying your hand calculations.